In [1]:
# Import the necessary libaries
import pandas as pd
import numpy as np
import tensorflow as tf
import re
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from transformers import AutoTokenizer
from datasets import load_dataset, DatasetDict


In [2]:
# Load and check the structure of a dataset
ds = load_dataset("KisanVaani/agriculture-qa-english-only")

print(ds["train"][0])

README.md:   0%|          | 0.00/1.74k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 1.97MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/22615 [00:00<?, ? examples/s]

{'question': 'why is crop rotation important in farming?', 'answers': 'This helps to prevent soil erosion and depletion, and can also help to control pests and diseases'}


In [3]:
# Convert the dataset into panda dataframe for easy preprocessing
df = pd.DataFrame(ds["train"])

In [4]:
# Check for the missing values
print(df.isnull().sum())

question    0
answers     0
dtype: int64


In [5]:
# Double check to drop the missing values even though there are not present
df.dropna(inplace=True)



In [6]:
# Remove special characters, extraspaces and lowercase all letters
def clean_text(text):
    text = text.lower()  # Convert to lowercase
    text = re.sub(r"[^a-zA-Z0-9?.!,']", " ", text)  # Remove special characters
    text = re.sub(r"\s+", " ", text).strip()  # Remove extra spaces
    return text

df['question'] = df['question'].apply(clean_text)
df['answer'] = df['answers'].apply(clean_text)


In [7]:
# Tokenize the model using wordPiece tokenization for BERT

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

# Tokenize the dataset
def tokenize_function(examples):
    return tokenizer(examples["question"], examples["answers"], padding="max_length", truncation=True)

tokenized_ds = ds.map(tokenize_function, batched=True)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/22615 [00:00<?, ? examples/s]

In [8]:
# Get the tokenized samples

print(tokenized_ds)
print(tokenized_ds['train'][0])

DatasetDict({
    train: Dataset({
        features: ['question', 'answers', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 22615
    })
})
{'question': 'why is crop rotation important in farming?', 'answers': 'This helps to prevent soil erosion and depletion, and can also help to control pests and diseases', 'input_ids': [101, 2339, 2003, 10416, 9963, 2590, 1999, 7876, 1029, 102, 2023, 7126, 2000, 4652, 5800, 14173, 1998, 2139, 10814, 3508, 1010, 1998, 2064, 2036, 2393, 2000, 2491, 20739, 2015, 1998, 7870, 102, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,

In [12]:
!pip install -q scikit-learn

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

questions = ds["train"]["question"]
answers = ds["train"]["answers"]

vectorizer = TfidfVectorizer(ngram_range=(1, 2), stop_words="english")
q_matrix = vectorizer.fit_transform(questions)

def ask(query):
    scores = cosine_similarity(vectorizer.transform([query]), q_matrix)[0]
    best = scores.argmax()
    return answers[best], float(scores[best])

print(ask("why is crop rotation important?"))

(' help to maintain soil health and improve crop yields over time', 1.0)


In [14]:
# Convert dataset to TensorFlow format

train_features = {key: tf.constant(tokenized_ds["train"][key]) for key in tokenizer.model_input_names}
train_labels = tf.constant([0] * len(tokenized_ds["train"]["question"]))  # Dummy labels for now

# Create TensorFlow dataset
train_dataset = tf.data.Dataset.from_tensor_slices((train_features, train_labels))
train_dataset = train_dataset.batch(8).shuffle(1000)


In [19]:
%pip install -q scikit-learn datasets

from datasets import load_dataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Load the dataset (no dependency on earlier cells)
ds = load_dataset("KisanVaani/agriculture-qa-english-only")["train"]
questions = ds["question"]
answers = ds["answers"]

# Build the TF-IDF index over the questions
vectorizer = TfidfVectorizer(ngram_range=(1, 2), stop_words="english")
q_matrix = vectorizer.fit_transform(questions)

def ask(query, threshold=0.15):
    scores = cosine_similarity(vectorizer.transform([query]), q_matrix)[0]
    best = int(scores.argmax())
    if scores[best] < threshold:
        return "Sorry, I couldn't find a good answer.", float(scores[best])
    return answers[best], float(scores[best])

print(ask("why is crop rotation important?"))

(' help to maintain soil health and improve crop yields over time', 1.0)


In [21]:
# Extract questions and answers
questions = ds["question"]
answers = ds["answers"]

In [22]:
print(questions)

Column(['why is crop rotation important in farming?', 'What farming practice helps prevent soil erosion?', 'what is crop rotation', 'what are the different methods of irrigation?', 'why is soil health vital?', ...])
